In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


#Configuration Of Model

In [ ]:
GPT_CONFIG_124M={
    "vocab_size":50257,
    "context_length":1024,
    "emb_dim":768,
    "n_heads":12,
    "n_layers":12,
    "drop_rate":0.1,
    "qkv_bias":False
}

#Transformer Block


In [ ]:
import torch
import torch.nn as nn
import math


class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

class MultiHeadAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, n_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % n_heads == 0)
        self.d_out = d_out
        self.num_heads = n_heads
        self.head_dim = d_out // n_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)

        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)

        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )

        attn_weights = torch.softmax(
            attn_scores / (self.head_dim ** 0.5),
            dim=-1
        )

        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)

        return context_vec

class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(math.sqrt(2 / math.pi) *(x + 0.044715 * x.pow(3))))

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            n_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"]
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])
    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        return x

#GPT MODEL

In [ ]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)
    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

In [ ]:
torch.manual_seed(123)
x = torch.randint(0, GPT_CONFIG_124M["vocab_size"], (2, 4))
gpt = GPTModel(GPT_CONFIG_124M)
op = gpt(x)
print("Input shape:", x.shape)
print("Output shape:", op.shape)

Input shape: torch.Size([2, 4])
Output shape: torch.Size([2, 4, 50257])


In [ ]:
!pip install torchinfo

In [ ]:
from torchinfo import summary
model=GPTModel(GPT_CONFIG_124M)
summary(model)

Layer (type:depth-idx)                   Param #
GPTModel                                 --
├─Embedding: 1-1                         38,597,376
├─Embedding: 1-2                         786,432
├─Dropout: 1-3                           --
├─Sequential: 1-4                        --
│    └─TransformerBlock: 2-1             --
│    │    └─MultiHeadAttention: 3-1      2,360,064
│    │    └─FeedForward: 3-2             4,722,432
│    │    └─LayerNorm: 3-3               1,536
│    │    └─LayerNorm: 3-4               1,536
│    │    └─Dropout: 3-5                 --
│    └─TransformerBlock: 2-2             --
│    │    └─MultiHeadAttention: 3-6      2,360,064
│    │    └─FeedForward: 3-7             4,722,432
│    │    └─LayerNorm: 3-8               1,536
│    │    └─LayerNorm: 3-9               1,536
│    │    └─Dropout: 3-10                --
│    └─TransformerBlock: 2-3             --
│    │    └─MultiHeadAttention: 3-11     2,360,064
│    │    └─FeedForward: 3-12            4,722,432
│   

In [ ]:
tp=sum(p.numel() for p in gpt.parameters())
print(f"Total parameters: {tp}")
tp1=tp-sum(p.numel() for p in gpt.out_head.parameters())
print("total parameters without out_head",tp1)

Total parameters: 163009536
total parameters without out_head 124412160


In [ ]:
#Approx Size
total_size=tp*4
total_size_mb=total_size/pow(1024,2)
print(f"total size of the model: {total_size_mb:.2f} MB")

total size of the model: 621.83 MB


#Loading Dataset And Tokenization

In [ ]:
with open("the-verdict.txt","r",encoding="utf-8") as f:
    text_data=f.read()


In [ ]:
print(len(text_data))

20479


In [ ]:
import tiktoken

def text_to_token(text,tokenizer):
  encoded=tokenizer.encode(text,allowed_special={'<|endoftext|>'})
  encoded_tensor=torch.tensor(encoded).unsqueeze(0)
  return encoded_tensor
def token_ids_to_text(token_ids,tokenizer):
  flat=token_ids.squeeze(0)
  return tokenizer.decode(flat.tolist())

In [ ]:
def generate_text_simple(model, idx, max_new_tokens,context_size):
  for _ in range(max_new_tokens):
    idx_cond=idx[:,-context_size:]
    with torch.no_grad():
      logits=model(idx_cond)
    logits=logits[:,-1,:]
    probas=torch.softmax(logits,dim=-1)
    idx_next=torch.argmax(probas,dim=1,keepdim=True)
  idx=torch.cat((idx,idx_next),dim=1)
  return idx

In [ ]:
tokenizer=tiktoken.get_encoding("gpt2")
s="Hello, I am"
enc=tokenizer.encode(s)
encoded_tensor=torch.tensor(enc).unsqueeze(0)
print(encoded_tensor.shape)

torch.Size([1, 4])


In [ ]:
model.eval()
out=generate_text_simple(
    model=gpt,
    idx=encoded_tensor,
    max_new_tokens=6,
    context_size=GPT_CONFIG_124M["context_length"]
)
print(out)

tensor([[15496,    11,   314,   716, 44798]])


#Creating Data Loaders

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import tiktoken

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.output_ids = []

        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1:i + 1 + max_length]

            self.input_ids.append(torch.tensor(input_chunk))
            self.output_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.output_ids[idx]


def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128,
                         shuffle=True, drop_last=True, num_workers=0):

    tokenizer = tiktoken.get_encoding("gpt2")

    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [ ]:
total_tokens=len(tokenizer.encode(text_data))
print(total_tokens)
print(len(text_data))

5145
20479


In [ ]:
GPT_CONFIG_124M['context_length']=256
train_ratio=0.90
split_idx=int(train_ratio*len(text_data))
train_data=text_data[:split_idx]
val_data=text_data[split_idx:]
train_loader=create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    shuffle=True,
    drop_last=True,
    num_workers=0
)
val_loader=create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    shuffle=True,
    drop_last=True,
    num_workers=0
)

In [ ]:
for x,y in train_loader:
  print(x.shape,y.shape)

torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])


#Defining Loss Function


In [ ]:
def cal_loss_batch(input_batch,target_batch,model,device):
  input_batch,target_batch=input_batch.to(device),target_batch.to(device)
  logits=model(input_batch)
  loss=torch.nn.functional.cross_entropy(logits.flatten(0,1),target_batch.flatten())
  return loss
def cal_loss_loader(data_loader,model,device,num_batches=None):
  total_loss=0
  num_batches=len(data_loader) if num_batches is None else min(num_batches,len(data_loader))
  for i,(input_batch,target_batch) in enumerate(data_loader):
    if i<num_batches:
      loss=cal_loss_batch(input_batch,target_batch,model,device)
      total_loss+=loss.item()
    else:
      break
  return total_loss/num_batches

#Training Loop

In [ ]:
def generate_text_simple(model, idx, max_new_tokens,context_size):
  for _ in range(max_new_tokens):
    idx_cond=idx[:,-context_size:]
    with torch.no_grad():
      logits=model(idx_cond)
    logits=logits[:,-1,:]
    probas=torch.softmax(logits,dim=-1)
    idx_next=torch.argmax(probas,dim=1,keepdim=True)
    idx=torch.cat((idx,idx_next),dim=1)
  return idx
def generate_and_print_sample(model,tokenizer,device,start_context):
  model.eval()
  context_size=model.pos_emb.weight.shape[0]
  encoded=text_to_token(start_context,tokenizer).to(device)
  with torch.no_grad():
    token_ids=generate_text_simple(
        model=model,idx=encoded,
        max_new_tokens=50,context_size=context_size
    )
  decoded_text = token_ids_to_text(token_ids, tokenizer)
  print(decoded_text.replace("\n", " "))
  model.train()

In [ ]:
def evaluate_model(model,train_loader,val_loader,device,eval_iter):
  model.eval()
  with torch.no_grad():
    train_loss=cal_loss_loader(train_loader,model,device,num_batches=eval_iter)
    val_loss=cal_loss_loader(val_loader,model,device,num_batches=eval_iter)
  model.train()
  return train_loss,val_loss


In [ ]:
def train_model_simple(model,train_loader,val_loader,optimizer,device,num_epochs,
                       eval_freq,eval_iter,start_context,tokenizer):

  train_losses,val_losses,track_tokens_seen=[],[],[]
  tokens_seen,global_step=0,-1

  for epoch in range(num_epochs):
    model.train()

    for input_batch,target_batch in train_loader:
      optimizer.zero_grad()
      loss=cal_loss_batch(input_batch,target_batch,model,device)
      loss.backward()
      optimizer.step()
      tokens_seen+=input_batch.numel()
      global_step+=1

      if global_step % eval_freq==0:
        train_loss,val_loss=evaluate_model(
            model,train_loader,val_loader,device,eval_iter
        )
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        track_tokens_seen.append(tokens_seen)
        print(f"Ep {epoch+1} (step {global_step:06d}):"
        f"Train loss {train_loss:.4f}, Val loss {val_loss:.4f}")
    generate_and_print_sample(
      model, tokenizer, device, start_context
    )
  return train_losses,val_losses,track_tokens_seen


In [ ]:
device="cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [ ]:

import time
start_time = time.time()

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)

num_epochs = 10
train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context="Every effort moves you", tokenizer=tokenizer
)

# Note:
# Uncomment the following code to show the execution time
end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

Ep 1 (step 000000):Train loss 9.8181, Val loss 9.9303
Ep 1 (step 000005):Train loss 7.9202, Val loss 8.3364
Every effort moves you,,,,,,,,,,,,.                                     
Ep 2 (step 000010):Train loss 6.5853, Val loss 7.0459
Ep 2 (step 000015):Train loss 5.9842, Val loss 6.5975
Every effort moves you, the, and, the, the, the, and, the. ", the,,, the, and, the,, the,, the, and, the, the,, the, and,,,,, the
Ep 3 (step 000020):Train loss 15.8715, Val loss 15.9469
Ep 3 (step 000025):Train loss 5.5781, Val loss 6.4526
Every effort moves you. Gis. Gis. Gis. Gis. G. I had to----, and--. I had to--. I had. I had to the his--. I had the to the ", and I had.
Ep 4 (step 000030):Train loss 5.0085, Val loss 6.3354
Ep 4 (step 000035):Train loss 4.7669, Val loss 6.2489
Every effort moves you, and I had been the, I had to the picture. ", I had been--I, I had been the to the of the, I had the to the, I had been, I had the picture. ", I
Ep 5 (step 000040):Train loss 4.2220, Val loss 6.2361
Eve

#Controlling Randomness Of Output

In [ ]:
tokenizer=tiktoken.get_encoding("gpt2")
token_ids=generate_text_simple(
    model=model,
    idx=text_to_token("Every effort moves you",tokenizer).to(device),
    max_new_tokens=25,
    context_size=GPT_CONFIG_124M["context_length"]
)
print(token_ids_to_text(token_ids,tokenizer))

Every effort moves you?"

"Yes--quite insensible to the irony. She wanted him vindicated--and by me!"




#Merging Temperature Scaling and TOP-K Sampling
logits->topk->divide by temp->softmax->multinomial

In [ ]:
def generate(model,idx,max_new_tokens,context_size,temp=0.0,top_k=None,eos_id=None):
  for _ in range(max_new_tokens):
    idx_cond=idx[:,-context_size:]
    with torch.no_grad():
      logits=model(idx_cond)
    logits=logits[:,-1,:]
    if top_k is not None:
      t_logits,_=torch.topk(logits,k=top_k)
      mini_val=t_logits[:,-1]
      logits=torch.where(logits<mini_val,torch.tensor(float("-inf")),logits)
    if temp>0:
      logits=logits/temp
      probs=torch.softmax(logits,dim=-1)
      idx_next=torch.multinomial(probs,num_samples=1)
    else:
      idx_next=torch.argmax(logits,dim=-1,keepdim=True)
    if idx_next==eos_id:
      break
    idx=torch.cat((idx,idx_next),dim=1)
  return idx


In [ ]:
torch.manual_seed(123)
token_ids=generate(
    model=model,
    idx=text_to_token("Every effort moves you",tokenizer).to(device),
    max_new_tokens=15,
    context_size=GPT_CONFIG_124M["context_length"],
    top_k=25,
    temp=1.6,
)
print(token_ids_to_text(token_ids,tokenizer))

Every effort moves you was "It his historyrow a for nothing--ituary handsome--and


#Storing and Loading Weights

In [ ]:
model=GPTModel(GPT_CONFIG_124M)
torch.save(model.state_dict(),"model.pth")

In [ ]:
model=GPTModel(GPT_CONFIG_124M)
model.load_state_dict(torch.load("model.pth"))

<All keys matched successfully>

In [ ]:
optimizer=torch.optim.AdamW(model.parameters(),lr=0.0004,weight_decay=0.1)
torch.save({
    "model_state_dict":model.state_dict(),
    "optimizer_state_dict":optimizer.state_dict(),
    },
    "model_and_optimizer.pth"
)

In [ ]:
check=torch.load("model_and_optimizer.pth")
model=GPTModel(GPT_CONFIG_124M)
model.load_state_dict(check["model_state_dict"])
optimizer=torch.optim.AdamW(model.parameters(),lr=5e-4,weight_decay=0.1)
optimizer.load_state_dict(check["optimizer_state_dict"])

#Loading OpenAI Weights


In [ ]:
from gpt3_download import download_and_load_gpt2
allowed_sizes = ("124M", "355M", "774M", "1558M")
settings,params=download_and_load_gpt2(model_size="124M",models_dir="gpt2")

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
checkpoint: 100%|██████████| 77.0/77.0 [00:00<00:00, 105kiB/s]
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
encoder.json: 100%|██████████| 1.04M/1.04M [00:01<00:00, 632kiB/s]
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verificati

In [ ]:
def assign(left,right):
  if left.shape != right.shape:
    return ValueError(f"Shape MisMatch")
  return torch.nn.Parameter(torch.tensor(right))

In [ ]:
import numpy as np

def load_weights_into_gpt(gpt, params):
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params['wpe'])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params['wte'])

    for b in range(len(params["blocks"])):
        q_w, k_w, v_w = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["w"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.weight = assign(
            gpt.trf_blocks[b].att.W_query.weight, q_w.T)
        gpt.trf_blocks[b].att.W_key.weight = assign(
            gpt.trf_blocks[b].att.W_key.weight, k_w.T)
        gpt.trf_blocks[b].att.W_value.weight = assign(
            gpt.trf_blocks[b].att.W_value.weight, v_w.T)

        q_b, k_b, v_b = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["b"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.bias = assign(
            gpt.trf_blocks[b].att.W_query.bias, q_b)
        gpt.trf_blocks[b].att.W_key.bias = assign(
            gpt.trf_blocks[b].att.W_key.bias, k_b)
        gpt.trf_blocks[b].att.W_value.bias = assign(
            gpt.trf_blocks[b].att.W_value.bias, v_b)

        gpt.trf_blocks[b].att.out_proj.weight = assign(
            gpt.trf_blocks[b].att.out_proj.weight,
            params["blocks"][b]["attn"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].att.out_proj.bias = assign(
            gpt.trf_blocks[b].att.out_proj.bias,
            params["blocks"][b]["attn"]["c_proj"]["b"])

        gpt.trf_blocks[b].ff.layers[0].weight = assign(
            gpt.trf_blocks[b].ff.layers[0].weight,
            params["blocks"][b]["mlp"]["c_fc"]["w"].T)
        gpt.trf_blocks[b].ff.layers[0].bias = assign(
            gpt.trf_blocks[b].ff.layers[0].bias,
            params["blocks"][b]["mlp"]["c_fc"]["b"])
        gpt.trf_blocks[b].ff.layers[2].weight = assign(
            gpt.trf_blocks[b].ff.layers[2].weight,
            params["blocks"][b]["mlp"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].ff.layers[2].bias = assign(
            gpt.trf_blocks[b].ff.layers[2].bias,
            params["blocks"][b]["mlp"]["c_proj"]["b"])

        gpt.trf_blocks[b].norm1.scale = assign(
            gpt.trf_blocks[b].norm1.scale,
            params["blocks"][b]["ln_1"]["g"])
        gpt.trf_blocks[b].norm1.shift = assign(
            gpt.trf_blocks[b].norm1.shift,
            params["blocks"][b]["ln_1"]["b"])
        gpt.trf_blocks[b].norm2.scale = assign(
            gpt.trf_blocks[b].norm2.scale,
            params["blocks"][b]["ln_2"]["g"])
        gpt.trf_blocks[b].norm2.shift = assign(
            gpt.trf_blocks[b].norm2.shift,
            params["blocks"][b]["ln_2"]["b"])

    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["g"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["b"])
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte"])

In [ ]:
new_config=GPT_CONFIG_124M.copy()
new_config.update({"context_length":1024,"qkv_bias":True})
gpt=GPTModel(new_config)
load_weights_into_gpt(gpt,params)
gpt.to(device);

In [ ]:
token_ids=generate(
    model=gpt,
    idx=text_to_token("Virat country",tokenizer).to(device),
    max_new_tokens=50,
    context_size=new_config["context_length"],
    top_k=30,
    temp=0.8
)
print(token_ids_to_text(token_ids,tokenizer))

Virat country, and to the people of Iraq, as well as the country of the people of the United States.

The first order of business is to make it easier for the people, the people, the people, and the people of the United States


#Fine Tuning

# 1. Classification Based Fine Tuning

In [ ]:
import pandas as pd

df = pd.read_csv(
    "SMSSpamCollection",
    sep="\t",
    header=None,
    names=["label", "message"]
)
print(df.head())
print((df["label"] == "ham").sum())
print((df["label"] == "spam").sum())

  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...
4825
747


In [ ]:
def create_balanced_df(df):
    num_samples_per_class = df["label"].value_counts().min()
    spam_subset = df[df["label"] == "spam"].sample(
        num_samples_per_class,
        random_state=123
    )
    ham_subset = df[df["label"] == "ham"].sample(
        num_samples_per_class,
        random_state=123
    )
    return pd.concat([spam_subset, ham_subset])

In [ ]:
balanced_df=create_balanced_df(df)
print((balanced_df["label"]=="ham").sum())

747


In [ ]:
balanced_df["label"]=balanced_df["label"].map({"ham":0,"spam":1})

In [ ]:
def random_split(df,train_frac,validation_frac):
  df=df.sample(frac=1,random_state=129).reset_index(drop=True)
  train_end=int(train_frac*len(df))
  val_end=train_end+int(validation_frac*len(df))
  train_df=df[:train_end]
  val_df=df[train_end:val_end]
  test_df=df[val_end:]
  return train_df,val_df,test_df
train_df,val_df,test_df=random_split(balanced_df,0.7,0.1)

In [ ]:
len(train_df)
train_df.head()

,label,message
0,0,"Good afternoon, my love! How goes that day ? I..."
1,1,December only! Had your mobile 11mths+? You ar...
2,0,He's an adult and would learn from the experie...
3,0,May b approve panalam...but it should have mor...
4,0,&lt;#&gt; ISH MINUTES WAS 5 MINUTES AGO. WTF.


#Creating DataLoaders


In [ ]:
import torch
from torch.utils.data import Dataset

class SpamDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=None, pad_token=50256):
        self.data=df

        self.encoded_texts = [
            tokenizer.encode(text) for text in self.data["message"]
        ]

        if max_length is None:
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length

            self.encoded_texts = [
                encoded_text[:self.max_length]
                for encoded_text in self.encoded_texts
            ]

        self.encoded_texts = [
            encoded_text + [pad_token] * (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]

    def __getitem__(self, index):
        encoded = self.encoded_texts[index]
        label = self.data.iloc[index]["label"]

        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(label, dtype=torch.long)
        )

    def __len__(self):
        return len(self.data)

    def _longest_encoded_length(self):
        return max(len(x) for x in self.encoded_texts)

In [ ]:
from numpy.random import test
train_dataset=SpamDataset(
    df=train_df,
    max_length=None,
    tokenizer=tokenizer,
)
val_dataset=SpamDataset(
    val_df,
    max_length=train_dataset.max_length,
    tokenizer=tokenizer,
)
test_dataset=SpamDataset(
    test_df,
    max_length=train_dataset.max_length,
    tokenizer=tokenizer,
)
print(train_dataset.max_length)

120


In [ ]:
from torch.utils.data import DataLoader
num_workers=0
batch_size=8
torch.manual_seed(123)
train_loader=DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True,
)
val_loader=DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    drop_last=False,
)
test_loader=DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    drop_last=False,
)

In [ ]:
for ip,t in train_loader:
  pass
print(ip.shape)
print(t.shape)
print(len(train_loader))

torch.Size([8, 120])
torch.Size([8])
130


#Loading OpenAI Weights

In [ ]:
from gpt3_download import download_and_load_gpt2
new_config=GPT_CONFIG_124M.copy()
new_config.update({"context_length":1024,"qkv_bias":True})
model=GPTModel(new_config)
settings,params=download_and_load_gpt2(model_size="124M",models_dir="gpt2")
load_weights_into_gpt(model,params)
model.eval();

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


File already exists and is up-to-date: gpt2/124M/checkpoint


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


File already exists and is up-to-date: gpt2/124M/encoder.json


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


...

File already exists and is up-to-date: gpt2/124M/model.ckpt.index


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


File already exists and is up-to-date: gpt2/124M/model.ckpt.meta


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


File already exists and is up-to-date: gpt2/124M/vocab.bpe


In [ ]:
model=model.to(device)

In [ ]:
text="every effort moves you"
token_ids=generate(
    model=model,
    idx=text_to_token(text,tokenizer).to(device),
    max_new_tokens=50,
    context_size=new_config["context_length"],
    top_k=30,
    temp=0.2
)
print(token_ids_to_text(token_ids,tokenizer))


every effort moves you to the next step.

The next step is to make sure you have a good understanding of what you're doing.

You'll notice that you're not always able to do this.

You may be able to do this by


In [ ]:
for param in model.parameters():
  param.requires_grad=False

In [ ]:
torch.manual_seed(123)
num_classes=2
model.out_head=torch.nn.Linear(in_features=GPT_CONFIG_124M["emb_dim"],out_features=num_classes)

In [ ]:
for param in model.trf_blocks[-1].parameters():
  param.requires_grad=True
for param in model.final_norm.parameters():
  param.requires_grad=True

#Accuracy

In [ ]:
def calc_accuracy_loader(data_loader, model, device, num_batches=None):
    model.eval()
    correct_predictions, num_examples = 0, 0
    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            input_batch, target_batch = input_batch.to(device), target_batch.to(device)
            with torch.no_grad():
                logits = model(input_batch)[:, -1, :]
            predicted_labels = torch.argmax(logits, dim=-1)

            num_examples += predicted_labels.shape[0]
            correct_predictions += (predicted_labels == target_batch).sum().item()
        else:
            break
    return correct_predictions / num_examples

In [ ]:
model.to(device)
train_acc=calc_accuracy_loader(train_loader,model,device,num_batches=10)
val_acc=calc_accuracy_loader(val_loader,model,device,num_batches=10)
test_acc=calc_accuracy_loader(test_loader,model,device,num_batches=10)
print(f"Train Acc:{train_acc*100:.2f}%")
print(f"Val Acc:{val_acc*100:.2f}%")
print(f"test acc:{test_acc*100:.2f}%")

Train Acc:42.50%
Val Acc:51.25%
test acc:56.25%


#Loss Function

In [ ]:
def calc_loss_batch(input_batch,target_batch,model,device):
  input_batch,target_batch=input_batch.to(device),target_batch.to(device)
  logits=model(input_batch)[:,-1,:]
  loss=torch.nn.functional.cross_entropy(logits,target_batch)
  return loss

In [ ]:
def calc_loss_loader(data_loader,model,device,num_batches=None):
  total_loss=0
  if len(data_loader)==0:
    return float("nan")
  elif num_batches is None:
    num_batches=len(data_loader)
  else:
    num_batches=min(num_batches,len(data_loader))
  for i,(input_batch,target_batch) in enumerate(data_loader):
    if i<num_batches:
      loss=calc_loss_batch(input_batch,target_batch,model,device)
      total_loss+=loss.item()
    else:
      break
  return total_loss/num_batches

In [ ]:
with torch.no_grad():
  train_loss=calc_loss_loader(train_loader,model,device,num_batches=5)
  val_loss=calc_loss_loader(val_loader,model,device,num_batches=5)
  test_loss=calc_loss_loader(test_loader,model,device,num_batches=5)
print(f"Train Loss:{train_loss:.3f}")
print(f"Val Loss:{val_loss:.3f}")
print(f"Test Loss:{test_loss:3f}")

Train Loss:2.561
Val Loss:2.569
Test Loss:1.918999


#Training Loop

In [ ]:
def train_classifier_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                            eval_freq, eval_iter):
    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    examples_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        model.train()

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()
            examples_seen += input_batch.shape[0]
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=eval_iter)
        val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=eval_iter)
        print(f"Training accuracy: {train_accuracy*100:.2f}% | ", end="")
        print(f"Validation accuracy: {val_accuracy*100:.2f}%")
        train_accs.append(train_accuracy)
        val_accs.append(val_accuracy)

    return train_losses, val_losses, train_accs, val_accs, examples_seen

In [ ]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

In [ ]:
import time

start_time = time.time()

torch.manual_seed(123)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)

num_epochs = 10
train_losses, val_losses, train_accs, val_accs, examples_seen = train_classifier_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=50, eval_iter=5,
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

Ep 1 (Step 000000): Train loss 0.453, Val loss 0.423
Ep 1 (Step 000050): Train loss 0.441, Val loss 0.444
Ep 1 (Step 000100): Train loss 0.318, Val loss 0.346
Training accuracy: 90.00% | Validation accuracy: 92.50%
Ep 2 (Step 000150): Train loss 0.402, Val loss 0.335
Ep 2 (Step 000200): Train loss 0.221, Val loss 0.350
Ep 2 (Step 000250): Train loss 0.335, Val loss 0.366
Training accuracy: 75.00% | Validation accuracy: 87.50%
Ep 3 (Step 000300): Train loss 0.320, Val loss 0.312
Ep 3 (Step 000350): Train loss 0.307, Val loss 0.254
Training accuracy: 85.00% | Validation accuracy: 92.50%
Ep 4 (Step 000400): Train loss 0.332, Val loss 0.231
Ep 4 (Step 000450): Train loss 0.192, Val loss 0.164
Ep 4 (Step 000500): Train loss 0.128, Val loss 0.131
Training accuracy: 92.50% | Validation accuracy: 92.50%
Ep 5 (Step 000550): Train loss 0.149, Val loss 0.200
Ep 5 (Step 000600): Train loss 0.105, Val loss 0.128
Training accuracy: 97.50% | Validation accuracy: 97.50%
Ep 6 (Step 000650): Train loss 

#Evaluating Model

In [ ]:
train_accuracy = calc_accuracy_loader(train_loader, model, device)
val_accuracy = calc_accuracy_loader(val_loader, model, device)
test_accuracy = calc_accuracy_loader(test_loader, model, device)

print(f"Training accuracy: {train_accuracy*100:.2f}%")
print(f"Validation accuracy: {val_accuracy*100:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")

Training accuracy: 98.37%
Validation accuracy: 95.97%
Test accuracy: 97.67%


In [ ]:
def classify_review(text, model, tokenizer, device, max_length=None, pad_token_id=50256):
    model.eval()
    input_ids = tokenizer.encode(text)
    supported_context_length = model.pos_emb.weight.shape[0]
    input_ids = input_ids[:min(max_length, supported_context_length)]
    input_ids += [pad_token_id] * (max_length - len(input_ids))
    input_tensor = torch.tensor(input_ids, device=device).unsqueeze(0)
    with torch.no_grad():
        logits = model(input_tensor)[:, -1, :]
    predicted_label = torch.argmax(logits, dim=-1).item()
    return "spam" if predicted_label == 1 else "not spam"

In [ ]:
text_1 = (
    "You are a winner you have been specially"
    " selected to receive $1000 cash or a $2000 award."
)

print(classify_review(
    text_1, model, tokenizer, device, max_length=train_dataset.max_length
))

spam


In [ ]:
torch.save(model.state_dict(),"spam_classifier.pth")
